# Faithfulness RAGAS 0.4.3
Model juri: Qwen3.8-27B INT4 yang sama dengan generator, dilayani vLLM (OpenAI API lokal), temperatur 0,
mode berpikir dimatikan. Prompt RAGAS bawaan tidak diubah. Langkah per jawaban sama dengan
`Faithfulness.ascore`: (1) pecah jawaban menjadi pernyataan, (2) NLI setiap pernyataan terhadap konteks
yang digabung dengan baris baru, (3) skor = pernyataan didukung / seluruh pernyataan.

In [ ]:
MODE = "full"
MODEL_ID = "RedHatAI/Qwen3.8-27B-INT4"
MODEL_REVISION = "e45c27e8f463817db788d141a73a3c235a0a7c0c"
CONCURRENCY = 32            # KV cache T4x2 ~75 ribu token; ~25-30 permintaan ~3 ribu token muat bersamaan
PRINT_EVERY = 256
MAX_TOKENS = 2048
TIME_BUDGET_S = 11 * 3600

In [ ]:
!pip install -q -U vllm 2>&1 | tail -2
!pip install -q ragas==0.4.3 langchain-community==0.4.1 2>&1 | tail -2   # 0.4.2 menghapus chat_models.vertexai yang diimpor ragas 0.4.3

In [ ]:
import asyncio, glob, json, math, os, platform, subprocess, time, urllib.request
os.environ["RAGAS_DO_NOT_TRACK"] = "true"
from importlib.metadata import version
import torch
t_start = time.time()
env = {k: version(k) for k in ("vllm", "torch", "ragas", "langchain-community", "instructor", "openai", "pydantic")}
env |= {"python": platform.python_version(), "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "mode": MODE}
print(json.dumps(env, indent=2))
json.dump(env, open("/kaggle/working/environment.json", "w"), indent=2)

In [ ]:
server_log = open("/kaggle/working/vllm_server.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL_ID, "--revision", MODEL_REVISION, "--served-model-name", "qwen", "--port", "8000",
     "--tensor-parallel-size", str(torch.cuda.device_count()), "--dtype", "float16", "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.92", "--seed", "0", "--enable-prefix-caching", "--kv-cache-dtype", "float16",
     "--max-num-batched-tokens", "2048", "--max-num-seqs", "32", "--enforce-eager", "--disable-custom-all-reduce",
     "--limit-mm-per-prompt", '{"image": 0, "video": 0}'],
    stdout=server_log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError(open("/kaggle/working/vllm_server.log").read()[-5000:])
        time.sleep(10)
print(f"server siap setelah {(time.time() - t_start) / 60:.1f} menit")

In [ ]:
def find(name):
    return sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
items = [json.loads(l) for l in open(find("faith_inputs.jsonl")[0], encoding="utf-8")]
if MODE == "smoke":
    keep = set(json.load(open(find("smoke_pids.json")[0])))
    items = [r for r in items if r["pid"] in keep]
out_path = "/kaggle/working/faithfulness.jsonl"
done = set()
for path in find("faithfulness.jsonl") + ([out_path] if os.path.exists(out_path) else []):
    done |= {json.loads(l)["pid"] for l in open(path, encoding="utf-8")}
todo = sorted((r for r in items if r["pid"] not in done), key=lambda r: r["pid"])
print(f"jawaban: {len(items)}, sudah: {len(done & {r['pid'] for r in items})}, sisa: {len(todo)}")

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness
client = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY", timeout=3600, max_retries=0)
llm = llm_factory("qwen", provider="openai", client=client, temperature=0.0, top_p=1.0, max_tokens=MAX_TOKENS,
                  seed=0, extra_body={"chat_template_kwargs": {"enable_thinking": False}})
metric = Faithfulness(llm=llm)
sem = asyncio.Semaphore(CONCURRENCY)

async def judge(r):
    # urutan langkah identik dengan Faithfulness.ascore; detail pernyataan disimpan untuk audit
    async with sem:
        if time.time() - t_start > TIME_BUDGET_S:
            return None                   # tidak memulai penilaian baru setelah batas waktu sesi
        t0 = time.time()
        rec = {"pid": r["pid"], "faithfulness": None, "n_statements": 0, "n_supported": 0,
               "statements": [], "verdicts": [], "error": None}
        try:
            statements = await metric._create_statements(r["question"], r["answer"])
            rec["statements"] = statements
            if statements:
                v = await metric._create_verdicts(statements, "\n".join(r["contexts"]))
                score = metric._compute_score(v)
                rec["verdicts"] = [{"statement": s.statement, "verdict": s.verdict, "reason": s.reason}
                                   for s in v.statements]
                rec["n_statements"] = len(v.statements)
                rec["n_supported"] = sum(1 for s in v.statements if s.verdict)
                rec["faithfulness"] = None if math.isnan(score) else score
        except Exception as e:
            rec["error"] = f"{type(e).__name__}: {e}"[:1000]
        rec["seconds"] = round(time.time() - t0, 2)
        return rec

async def run():
    n = err = 0
    with open(out_path, "a", encoding="utf-8") as f:
        for fut in asyncio.as_completed([judge(r) for r in todo]):   # antrean mengalir
            rec = await fut
            if rec is None:
                continue
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            n += 1; err += rec["error"] is not None
            if n % PRINT_EVERY == 0:
                print(f"{n}/{len(todo)}  galat {err}  total {(time.time() - t_start) / 60:.0f} menit", flush=True)
    print(f"selesai sesi ini: {n}/{len(todo)}, galat {err}; sisa {len(todo) - n}", flush=True)

await run()

In [ ]:
rows = [json.loads(l) for l in open(out_path, encoding="utf-8")]
vals = [r["faithfulness"] for r in rows if r["faithfulness"] is not None]
print(f"baris: {len(rows)}, bernilai: {len(vals)}, tanpa pernyataan: {sum(r['n_statements'] == 0 and r['error'] is None for r in rows)}, "
      f"galat: {sum(r['error'] is not None for r in rows)}, rata-rata: {sum(vals) / max(len(vals), 1):.4f}")
for r in rows[:3]:
    print("-" * 80); print(json.dumps(r, ensure_ascii=False, indent=1)[:2500])
for r in [r for r in rows if r["error"]][:3]:
    print("GALAT", r["pid"], r["error"][:500])
server.terminate()